# Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# Read from Silver tables

Proizvodi dolaze iz dva izvora:
- `crm_products` — naziv, cost, line, start_date, end_date  
- `erp_px_cat`   — category, subcategory, maintenance

**Vazna stvar:** `product_key` u CRM-u izgleda ovako: `CO-RF-FR-R92B-58`  
Prve dva segmenta (`CO`) su category_id koji se matchuje sa `erp_px_cat`.
Koristicemo `split` + `explode` logiku da izvucemo taj prefix.

In [0]:
crm_products = spark.table("workspace.silver.crm_products")
erp_px_cat   = spark.table("workspace.silver.erp_px_cat")

print("=== crm_products ===")
crm_products.printSchema()
crm_products.show(3, truncate=False)

print("=== erp_px_cat ===")
erp_px_cat.printSchema()
erp_px_cat.show(3, truncate=False)

# Business Transformation — dim_products

## explode / split

`product_key` = `'CO-RF-FR-R92B-58'`  
`split('-')[0]` = `'CO'` — to je category_id koji matchujemo sa `erp_px_cat`

Ovo je prakticna upotreba `split()` — ne treba nam `explode` ovde jer
uzimamo samo jedan element, ali princip je isti.

**Kada bi koristili `explode`?**  
Kada jedan red ima array vrednosti i hoces svaku u poseban red:
```python
# Primer: tags = ['sale', 'premium', 'new']
df.withColumn('tag', F.explode(F.col('tags')))
# rezultat: 3 reda umesto 1
```

## SCD Type 2 logika

Silver tabela vec ima `start_date` i `end_date` po proizvodu!  
Kada `end_date` je NULL — to je trenutno aktivan red (current record).  
Ovo je klasican **SCD Type 2** pattern: cuvamo istoriju izmena.

```
product_key | name      | cost | start_date | end_date   | is_current
FR-R92B-58  | Road Bike | 800  | 2020-01-01 | 2022-06-30 | false   <- stara cena
FR-R92B-58  | Road Bike | 950  | 2022-07-01 | NULL       | true    <- trenutna cena
```

Razlika od SCD Type 1 (dim_customers):  
- Type 1 = overwrite, nema istorije  
- Type 2 = novi red za svaku promenu, stari red dobija end_date

In [0]:
# Korak 1: Izvuci category_id iz product_key
# product_key format: 'CO-RF-FR-R92B-58'
# split po '-', uzmi prvi element = category prefix

crm_prep = crm_products.withColumn(
    "category_id",
    F.split(F.col("product_key"), "-")[0]
)

# Takodje ciscenje product_key: ukloni category prefix da dobijemo cist kljuc
# 'CO-RF-FR-R92B-58' -> 'RF-FR-R92B-58'
crm_prep = crm_prep.withColumn(
    "product_key_clean",
    F.regexp_replace(F.col("product_key"), r'^[A-Z]+-', '')
)

print("Category prefixes u podacima:")
crm_prep.select("product_key", "category_id", "product_key_clean").show(5, truncate=False)

In [0]:
# Korak 2: Pripremi category tabelu
cat_prep = erp_px_cat.select(
    F.col("category_id").alias("cat_id"),
    F.col("category"),
    F.col("subcategory"),
    F.col("maintenance")
)

In [0]:
# Korak 3: JOIN products + categories
joined = (
    crm_prep.select(
        F.col("product_id"),
        F.col("product_key_clean").alias("product_key"),
        F.col("name"),
        F.col("cost"),
        F.col("line"),
        F.col("start_date"),
        F.col("end_date"),
        F.col("category_id")
    )
    .join(
        cat_prep,
        crm_prep["category_id"] == cat_prep["cat_id"],
        how="left"
    )
)

In [0]:
# Korak 4: Finalna dim_products
# - is_current flag: end_date IS NULL = aktivan red
# - surrogate key
# - line normalizacija

window_spec = Window.orderBy("product_key", "start_date")

dim_products = (
    joined
    .withColumn(
        "product_key_sk",
        F.row_number().over(window_spec)
    )
    .withColumn(
        "is_current",
        F.col("end_date").isNull()
    )
    .withColumn(
        # Normalizacija product line kodova
        "line",
        F.when(F.col("line") == "M", "Mountain")
         .when(F.col("line") == "R", "Road")
         .when(F.col("line") == "S", "Other Sales")
         .when(F.col("line") == "T", "Touring")
         .otherwise("n/a")
    )
    .select(
        F.col("product_key_sk"),
        F.col("product_id"),
        F.col("product_key"),
        F.col("name"),
        F.col("cost"),
        F.col("line"),
        F.col("category"),
        F.col("subcategory"),
        F.col("maintenance"),
        F.col("start_date"),
        F.col("end_date"),
        F.col("is_current")
    )
)

dim_products.display()

In [0]:
# Provera SCD Type 2 — da li imamo isti product_key sa vise verzija?
# Ocekujemo da neki proizvodi imaju is_current=true I is_current=false redove
(
    dim_products
    .groupBy("product_key")
    .agg(
        F.count("*").alias("broj_verzija"),
        F.sum(F.col("is_current").cast("int")).alias("current_verzija")
    )
    .filter(F.col("broj_verzija") > 1)
    .orderBy(F.col("broj_verzija").desc())
    .show(10)
)

# Write to Gold — Initial Load

In [0]:
(
    dim_products.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("workspace.gold.dim_products")
)

# MERGE — SCD Type 2 upsert

Ovo je srce SCD Type 2 u Delta Lake-u.  
Kada stignu novi podaci (incrementalni load), ne radimo overwrite —  
radimo MERGE koji:

1. Ako red **postoji i nije se promenio** → nista  
2. Ako red **postoji ali se cost promenio** → zatvori stari red (postavi end_date), ubaci novi  
3. Ako red **ne postoji** → ubaci novi (INSERT)

```
MERGE INTO target
USING source
ON match_condition
WHEN MATCHED AND changed THEN UPDATE   <- zatvori stari
WHEN NOT MATCHED THEN INSERT            <- novi proizvod
```

**apply_changes** (DLT) je viši nivo abstrakcije — DLT sam upravlja
ovom logikom. MERGE ti daje punu kontrolu, apply_changes ti daje
jednostavnost.

In [0]:
%sql
-- Simulacija: dosao je novi batch sa izmenjenom cenom za jedan proizvod
-- U realnosti bi ovo doslo iz novog CSV-a ili streaming izvora

-- Provera trenutnog stanja pre merge-a
SELECT product_key, name, cost, start_date, end_date, is_current
FROM workspace.gold.dim_products
WHERE is_current = true
LIMIT 5

In [0]:
# MERGE — upsert logika za SCD Type 2
# Koristimo DeltaTable API

target = DeltaTable.forName(spark, "workspace.gold.dim_products")

# Novi podaci koji stizu (source)
# U realnosti bi ovo bio novi DataFrame iz Auto Loader / batch job-a
new_data = spark.sql("""
    SELECT product_key_sk, product_id, product_key, name, cost, line,
           category, subcategory, maintenance,
           start_date, end_date, is_current
    FROM workspace.gold.dim_products
    WHERE is_current = true
    LIMIT 3
""")

# SCD Type 1 MERGE (simple upsert — overwrite ako postoji, insert ako ne)
# Ovo je Type 1 jer update-ujemo posteci red
(
    target.alias("target")
    .merge(
        new_data.alias("source"),
        "target.product_key = source.product_key AND target.is_current = true"
    )
    .whenMatchedUpdate(set={
        "cost":       "source.cost",
        "line":       "source.line",
        "is_current": "source.is_current"
    })
    .whenNotMatchedInsertAll()
    .execute()
)

print("MERGE kompletiran.")

In [0]:
%sql
-- DESCRIBE HISTORY pokazuje MERGE kao poseban operation type
DESCRIBE HISTORY workspace.gold.dim_products

# Table Properties

Table properties su metadata koja se cuva u Delta logu.  
Koristimo ih za:
- dokumentaciju (`comment`)
- Delta ponasanje (`delta.logRetentionDuration`, `delta.deletedFileRetentionDuration`)
- custom business metadata (`owner`, `domain`, `pii`)

In [0]:
%sql
-- Dodavanje table properties
ALTER TABLE workspace.gold.dim_products
SET TBLPROPERTIES (
    'domain'                          = 'products',
    'scd_type'                        = '2',
    'pii'                             = 'false',
    'delta.logRetentionDuration'      = 'interval 30 days',
    'delta.deletedFileRetentionDuration' = 'interval 7 days'
)

In [0]:
%sql
-- Citanje table properties
DESCRIBE DETAIL workspace.gold.dim_products

In [0]:
%sql
-- Uklanjanje jedne property
ALTER TABLE workspace.gold.dim_products
UNSET TBLPROPERTIES ('pii')

# Managed vs External tables

**Managed table** (ono sto smo mi pravili):
```sql
df.write.saveAsTable("workspace.gold.dim_products")
```
- Databricks kontrolise i lokaciju podataka i metadata
- `DROP TABLE` brise i podatke i metadata
- Podaci idu u default Unity Catalog lokaciju

**External table**:
```sql
CREATE TABLE workspace.gold.dim_products_ext
LOCATION 'abfss://container@storage.dfs.core.windows.net/gold/dim_products'
```
- Ti kontrolises lokaciju podataka
- `DROP TABLE` brise samo metadata, podaci ostaju
- Koristis kada imas existing data lake ili deles podatke sa drugim sistemima

**Kada koje?**
- Managed → novi projekti u Databricksu, Unity Catalog
- External → legacy sistemi, deljenje podataka, specificne storage politike

In [0]:
%sql
-- Provera da li je tabela managed ili external
DESCRIBE EXTENDED workspace.gold.dim_products